In [1]:
import os
import sys

txpipe_dir = os.path.join(os.environ["HOME"], "TXPipe")
sys.path.append(txpipe_dir)

# os.chdir("/global/homes/k/kabelo/TXPipe")
if os.getcwd().endswith("notebooks"):
    os.chdir("..")
    
import matplotlib.pyplot as plt
import h5py
import numpy as np
from txpipe.data_types import HDFFile, ShearCatalog, FiducialCosmology
import sacc
import yaml
from pprint import pprint
import time
import ceci
from txpipe.twopoint import TXTwoPoint

%matplotlib inline

In [ ]:
class TXTwoPointRand(TXTwoPoint):
    name = "TXTwoPointRand"
    inputs = [
        ("random_catalog", HDFFile),
        ("shear_catalog", ShearCatalog),
        ("fiducial_cosmology", FiducialCosmology)
    ]
    
    outputs = [("twopoint_gamma_x", SACCFile)]
    
    config_options = {
          # "binning_scale": "Log"   # idk if i even need this
          "max_sep": 250.0, # Mpc
          "min_sep": 10.0, # Mpc
          "nbins": 15,
          "redshift_bin_edges": [0.4, 0.8, 1.2],
          "richness_bin_edges": [20, 30, 200],
          "units": "arcmin",
          "chunk_rows": 100000,
          "nside": 64
          "pixelization": healpix
          "sparse": True
    }

    def make_random_catalog(self, i):
        # As with the lens catalog version, we add the r_col keyword
        # compare to the parent class
        import treecorr

        if not self.config["use_randoms"]:
            return None

        rancat = treecorr.Catalog(
            self.get_input("binned_random_catalog"),
            ext=f"/randoms/bin_{i}",
            ra_col="ra",
            dec_col="dec",
            ra_units="degree",
            dec_units="degree",
            patch_centers=self.get_input("patch_centers"),
            save_patch_dir=self.get_patch_dir("binned_random_catalog", i),
        )
        return rancat

In [2]:
from txpipe.random_cats import TXRandomCat
from txpipe.auxiliary_maps import TXAuxiliaryLensMaps

In [53]:
photometry_cat = h5py.File('/global/homes/k/kabelo/TXPipe/data/example/inputs/photometry_catalog.hdf5', 'r')

In [43]:
outdir = "/global/homes/k/kabelo/TXPipe/data/example/outputs"
os.makedirs(outdir, exist_ok=True)

args = {
    "config": {},  # should just pick up the defaults
    "pixelization": "healpix",
    "nside": 2048,
    "photometry_catalog": "/global/homes/k/kabelo/TXPipe/data/example/inputs/photometry_catalog.hdf5",
    "aux_lens_maps": f"{outdir}/aux_maps.hdf5",
}

aux_stage = TXAuxiliaryLensMaps(args)

# Now run the stage
aux_stage.run()

/global/homes/k/kabelo/TXPipe/txpipe/utils/provenance.py:18: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  elif hasattr(module, "__version__"):
/global/homes/k/kabelo/TXPipe/txpipe/utils/provenance.py:19: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  v = module.__version__


In [54]:
outdir = "/global/homes/k/kabelo/TXPipe/data/example/outputs"
os.makedirs(outdir, exist_ok=True)

args = {
    'config': {},
    'aux_lens_maps': 'global/homes/k/kabelo/TXPipe/data/example/outputs_metadetect/aux_lens_maps.hdf5',
    'lens_photoz_stack': 'global/homes/k/kabelo/TXPipe/data/example/inputs/photometry_catalog.hdf5',
    'fiducial_cosmology': 'global/homes/k/kabelo/TXPipe/data/fiducial_cosmology.yml',
    "random_cats": f"{outdir}/random_cats.hdf5",
    "binned_random_catalog": f"{outdir}/binned_random_catalog.hdf5",
    "binned_random_catalog_sub": f"{outdir}/binned_random_catalog_sub.hdf5"
}

rancat_stage = TXRandomCat(args)
rancat_stage.run()

ValueError: 

TXRandomCat Missing these names on the command line:
    Input names: --mask

### Running metadetect pipeline so I have the right files available...

In [2]:
pipeline = ceci.Pipeline.read('./examples/metadetect/pipeline.yml')
pprint(pipeline)

Skipping stage TXAuxiliaryLensMaps because its outputs exist already
Skipping stage TXStarCatalogSplitter because its outputs exist already
Skipping stage TXPSFDiagnostics because its outputs exist already
Skipping stage TXBrighterFatterPlot because its outputs exist already
Skipping stage FlowCreator because its outputs exist already
Skipping stage TXSimpleMaskFrac because its outputs exist already
Skipping stage GridSelection because its outputs exist already
Skipping stage TXParqetToHDF because its outputs exist already
Skipping stage PZPrepareEstimatorLens because its outputs exist already
Skipping stage NZDirInformerLens because its outputs exist already
Skipping stage TXSourceSelectorMetadetect because its outputs exist already
Skipping stage NZDirInformerSource because its outputs exist already
Skipping stage PZEstimatorLens because its outputs exist already
Skipping stage TXShearCalibration because its outputs exist already
Skipping stage TXSourceNoiseMaps because its outputs e

In [3]:
pipeline.run()


Executing TXConvergenceMaps
Command is:
OMP_NUM_THREADS=1 PYTHONPATH=submodules/pyfsb:$PYTHONPATH  python3 -m txpipe TXConvergenceMaps   --source_maps=data/example/outputs_metadetect/source_maps.hdf5   --config=examples/metadetect/config.yml   --convergence_maps=data/example/outputs_metadetect/convergence_maps.hdf5 
Output writing to data/example/logs_metadetect/TXConvergenceMaps.out

Job TXConvergenceMaps has failed with status 1



*************************************************
Error running pipeline stage TXConvergenceMaps.
Failed after 2.0 seconds.

Standard output and error streams in data/example/logs_metadetect/TXConvergenceMaps.out
*************************************************


1

In [12]:
import sys
!{sys.executable} -m pip install git+https://github.com/LSSTDESC/WLMassMap.git

  Cloning https://github.com/LSSTDESC/WLMassMap.git to /tmp/pip-req-build-lafi6h4n
  Running command git clone --filter=blob:none --quiet https://github.com/LSSTDESC/WLMassMap.git /tmp/pip-req-build-lafi6h4n
  Resolved https://github.com/LSSTDESC/WLMassMap.git to commit 23ae399cbeb2142c449cd1da5f5fd258674a2a9e
  Preparing metadata (setup.py) ... one
  Preparing metadata (setup.py) ... one
  Created wheel for wlmassmap: filename=wlmassmap-0.0.1-py3-none-any.whl size=14055 sha256=4c46e3d0f05de2aff8b902658a9faff7676469553c6908075c135576ada118bb
  Stored in directory: /tmp/pip-ephem-wheel-cache-xd1jznnv/wheels/a0/a5/90/d95700a3affb168749bcf54ca384d566abf6da150846725acf
  Created wheel for descformats: filename=descformats-0.0.7-py3-none-any.whl size=5709 sha256=74988b1686c01fda82e563926a9a8c9917f775c1ada0a12fbe2869592f5f6ad4
  Stored in directory: /global/u2/k/kabelo/.cache/pip/wheels/24/95/f2/53339def013709cffbce81d6d4e5ef2fd205bd7f97ab5ea9b0
Successfully built wlmassmap descformats


In [10]:
sys.executable

'/global/cfs/projectdirs/lsst/groups/CL/cl_pipeline_project/conda_envs/txpipe_clp/bin/python'